# 特征空间分析

输出 `results/` 下的 `pc_association.csv`、`explained_variance.csv`、`rank_axis.csv` 和 `rank_axis/<model>.npy`。
每格自己读数据，可以单独运行。路径在 `fmspace/paths.py`。

## PC × 标签关联、解释方差

PCA 用全部 1,189 张拟合（不用标签）。

In [ ]:
import numpy as np
import pandas as pd

from fmspace.analysis import association, l2, pca
from fmspace.data import load_metadata
from fmspace.extract import load_features
from fmspace.models import MODELS
from fmspace.paths import DATA_DIR, FEATURE_DIR, RESULT_DIR

metadata = load_metadata(DATA_DIR)
assoc, variance = [], []
for name in MODELS:
    X = l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
    p = pca(X)
    assoc.append(association(p.transform(X), metadata).assign(model=name))
    variance.append(pd.DataFrame({"model": name, "component": np.arange(1, 51), "ratio": p.explained_variance_ratio_}))
pd.concat(assoc).to_csv(RESULT_DIR / "pc_association.csv", index=False)
pd.concat(variance).to_csv(RESULT_DIR / "explained_variance.csv", index=False)

## rank axis

按 Sonthalia et al. (NeurIPS 2025) 的流程训练。`rank_rho`、`pc1_rho`、`diff` 都在 seed 0 的测试集上计算，`diff_low`、`diff_high` 为患者 bootstrap 95% 区间。

In [ ]:
import numpy as np
import pandas as pd

from fmspace.analysis import compare_with_pc1, l2, pca, rank_axis
from fmspace.data import load_metadata, patient_split
from fmspace.extract import load_features
from fmspace.models import MODELS
from fmspace.paths import DATA_DIR, FEATURE_DIR, RESULT_DIR

metadata = load_metadata(DATA_DIR)
split = patient_split(metadata, seed=0)
y = metadata["eye_dr_level"].to_numpy()
test = (split == "test").to_numpy()
rows = []
for name in MODELS:
    X = l2(load_features(FEATURE_DIR / f"{name}.npz", metadata["image_id"]))
    r = rank_axis(X, y, split)
    np.save(RESULT_DIR / "rank_axis" / f"{name}.npy", r["weight"])
    score = X @ r["weight"]
    pc1 = pca(X).transform(X)[:, 0]
    rows.append(
        {"model": name, "lr": r["lr"], "weight_decay": r["weight_decay"], "val_rho": r["val_rho"],
         **compare_with_pc1(score[test], pc1[test], y[test], metadata["patient_id"][test])}
    )
summary = pd.DataFrame(rows)
summary.to_csv(RESULT_DIR / "rank_axis.csv", index=False)
summary